# SPARCL vs DESI spectra
The goal of this notebook is to compare, for an example spectrum, the file in the DESI catalogue against SPARCL. 
In particular, we are interested in how different the resolution matrix is from the gaussian spectral dispersion array provided by SPARCL.

In [ ]:
import desispec

In [ ]:
from astropy.table import Table

In [ ]:
from astropy.io import fits
from astropy.table import Table

In [ ]:
import numpy as np

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
from astropy.utils.data import download_file


In [ ]:
import arya
arya.init("mnras")

In [ ]:
import scipy.special

In [ ]:
from desispec.resolution import Resolution

In [ ]:
from sparcl.client import SparclClient

In [ ]:
from desi_utils.fit_lines import _gaussian_flux, _pixel_edges

## Data loading
here, we select a target id to retrieve. Unfortunately, we have to download the entire DESI fits file (400 Mb).

In [ ]:
dwarf_cat = Table.read("data/stellar_mass_emline_dwarfs.fits")

In [ ]:
dwarf_cat[0]

In [ ]:
targetid = dwarf_cat["targetid"][0]

In [ ]:
DESI_DATA_URL = "https://data.desi.lbl.gov/public/dr1/spectro/redux/iron/"

In [ ]:
def get_desi_path(record):
    survey = record["survey"]
    program = record["program"]
    pixnum = record["healpix"]
    pixgroup = pixnum // 100
    path = f"{DESI_DATA_URL}/healpix/{survey}/{program}/{pixgroup}/{pixnum}/coadd-{survey}-{program}-{pixnum}.fits"

    return path



In [ ]:
url = get_desi_path(dwarf_cat[0])

In [ ]:
filename = download_file(url, cache=True, show_progress=True)


In [ ]:
ff = fits.open(filename)


In [ ]:
tab = Table.read(ff[1])

In [ ]:
tab

In [ ]:
spec_idx = np.where(tab["TARGETID"] == targetid)[0][0]

In [ ]:
tab[spec_idx]

The structure of the HDF file
- HDU 00 contains the keywords
- HDU 1 contains useful properties of each object
- HDU 3-7 contain the B spectroscopic info in the order: wavelength, flux, inverse variance flux, mask of flux, resolution matrix
- HDU 8-12 for R
- HDU 13-17 for Z
- HDU 18, the scores for the template fitting.

## Retrieving the spectral information

In [ ]:
15707 * 410 / 1e3 # this is the amount of data we need to download in GB!

In [ ]:
def get_spectrum(ff, spec_idx, band = "b"):
    idx0 = {
        "b": 3, 
        "r": 8,
        "z": 13
    }[band]

    wave = ff[idx0].data
    flux = ff[idx0+1].data[spec_idx, :]
    ivar = ff[idx0+2].data[spec_idx, :]
    mask = ff[idx0+3].data[spec_idx, :]
    resol = ff[idx0+4].data[spec_idx, :, :]

    return wave, flux, ivar, mask, Resolution(resol)

In [ ]:
lam_b, flux_b, ivar_b, mask_b, resol_b = get_spectrum(ff, spec_idx, "b")
lam_r, flux_r, ivar_r, mask_r, resol_r = get_spectrum(ff, spec_idx, "r")
lam_z, flux_z, ivar_z, mask_z, resol_z = get_spectrum(ff, spec_idx, "z")

In [ ]:
client = SparclClient()

In [ ]:
def get_spectra(target_id):
    inc = ['sparcl_id', 'specid', 'data_release', 'redshift', 'flux',
       'wavelength', 'model', 'ivar', 'mask', 'spectype', 'ra', 'dec', 'wave_sigma']


    galspec_raw = client.retrieve_by_specid(specid_list = [target_id], 
        include = inc,
        dataset_list = ['DESI-DR1'], 
        fmt="specutils")

    return galspec_raw

In [ ]:
sparclspec = get_spectra(int(targetid))

# Plots of spectrum

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3))

plt.plot(lam_b, flux_b)
plt.plot(lam_r, flux_r)
plt.plot(lam_z, flux_z)
plt.plot(sparclspec.spectral_axis, sparclspec.flux, lw=0.5)

ax.set(xlabel = r"wavelength / $\AA$",
       ylabel = "flux")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3))

plt.plot(lam_b, ivar_b)
plt.plot(lam_r, ivar_r)
plt.plot(lam_z, ivar_z)

plt.plot(sparclspec.spectral_axis, sparclspec.uncertainty.array, lw=0.5)

ax.set(xlabel = r"wavelength / $\AA$",
       ylabel = "ivar flux")

In [ ]:
plt.plot(lam_b, mask_b)
plt.plot(lam_r, mask_r)
plt.plot(lam_z, mask_z)

plt.plot(sparclspec.spectral_axis, sparclspec.uncertainty.array == 0, lw=0.5)


ax.set(xlabel = r"wavelength / $\AA$",
       ylabel = "masked pixels")

# The resolution matrix

In [ ]:
R_b = resol_b.astype(np.float64).data
R_r = resol_r.astype(np.float64).data
R_z = resol_z.astype(np.float64).data

In [ ]:
fig, ax = plt.subplots(figsize=(10, 2))
plt.imshow(np.log10(R_b), interpolation="none", )

ax.set(
    aspect = 20
)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 2))
plt.imshow(np.log10(R_r), interpolation="none", )

ax.set(
    aspect = 20
)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 2))
plt.imshow(np.log10(R_z), interpolation="none", )

ax.set(
    aspect = 20
)

In [ ]:
def reconstruct_resolution(lam, sparclspec):
    resol_reconstructed = np.zeros((11, len(lam)))

    dx = np.diff(lam)[0]
    assert np.all(np.isclose(dx, np.diff(lam)))
    sigma_px = sparclspec.meta["wave_sigma"].value / dx 
    
    for i in range(len(lam)):
        xx = np.arange(-5, 6,)
        edges = _pixel_edges(xx)
    
        ii = np.argmin(np.abs(lam[i] - sparclspec.spectral_axis.value))
        
        y = _gaussian_flux(xx, 1, 0, sigma_px[ii], edges=edges, exp=np.exp, erf=scipy.special.erf)
        
        resol_reconstructed[:, i] = y / np.sum(y)

    return resol_reconstructed

In [ ]:
Rs_b = reconstruct_resolution(lam_b, sparclspec)
Rs_r = reconstruct_resolution(lam_r, sparclspec)
Rs_z = reconstruct_resolution(lam_z, sparclspec)

In [ ]:
Rs_b

In [ ]:
fig, ax = plt.subplots(figsize=(10, 2))
plt.imshow(np.log10(Rs_b), interpolation="none", )

ax.set(
    aspect = 20
)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 2))
plt.imshow(np.log10(Rs_r), interpolation="none", )

ax.set(
    aspect = 20
)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 2))
plt.imshow(np.log10(Rs_z), interpolation="none", )

ax.set(
    aspect = 20
)

In [ ]:

fig, ax = plt.subplots(figsize=(10, 2))

plt.plot(lam_b, np.sum(R_b, axis=0), lw=0.5)
plt.plot(lam_r, np.sum(R_r, axis=0), lw=0.5)
plt.plot(lam_z, np.sum(R_z, axis=0), lw=0.5)
plt.xlabel(r"wavelength / $\AA$")
plt.ylabel("flux from resolution matrix")

In [ ]:
from scipy.ndimage import gaussian_filter1d

In [ ]:
skyspec = Table.read("data/skyspectrum.fits")

In [ ]:
skyspec

In [ ]:

fig, ax = plt.subplots(figsize=(10, 2))
w = 3

plt.plot(lam_b, gaussian_filter1d(np.sum(R_b, axis=0), w), lw=0.5)
plt.plot(lam_r, gaussian_filter1d(np.sum(R_r, axis=0), w), lw=0.5)
plt.plot(lam_z, gaussian_filter1d(np.sum(R_z, axis=0), w), lw=0.5)
plt.plot(skyspec["wavelength"], skyspec["flux"] * 0.003)
plt.xlabel(r"wavelength / $\AA$")
plt.ylabel("flux from resolution matrix")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 2))
p = plt.imshow( (R_b - Rs_b), interpolation="none", cmap="RdBu", clim=(-0.1, 0.1),
              origin="lower", extent =(lam_b[0], lam_b[-1], -5, 5))

plt.colorbar(label="true - gaussian")
ax.set(
    aspect = 20,
    xlabel = r"wavelength / $\AA$",
    ylabel = "R",
)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 2))
p = plt.imshow( (R_r - Rs_r), interpolation="none", cmap="RdBu", clim=(-0.1, 0.1),
              origin="lower", extent =(lam_r[0], lam_r[-1], -5, 5))

plt.colorbar(label="true - gaussian")
ax.set(
    aspect = 20,
    xlabel = r"wavelength / $\AA$",
    ylabel = "R",
)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 2))
p = plt.imshow( (R_z - Rs_z), interpolation="none", cmap="RdBu", clim=(-0.1, 0.1),
              origin="lower", extent =(lam_z[0], lam_z[-1], -5, 5))

plt.colorbar(label="true - gaussian")
ax.set(
    aspect = 20,
    xlabel = r"wavelength / $\AA$",
    ylabel = "R",
)